Este notebook nos va a servir para hacer fine tunning del modelo, pero los más importante viene después. Vamos a hacer un estudio con pruebas para ver si es o no rentable hacer fine tunning de nuestro modelo de embeddings para el RAG, y porqué.

In [ ]:
!pip install -q sentence-transformers ebooklib beautifulsoup4 lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 4.2 MB/s eta 0:00:00


In [ ]:
import os
import torch
from pathlib import Path
from sentence_transformers import SentenceTransformer, InputExample, losses, util
from torch.utils.data import DataLoader
import ebooklib
import numpy as np
from ebooklib import epub
from bs4 import BeautifulSoup
from google.colab import drive
import nltk
from nltk.tokenize import sent_tokenize

In [ ]:
REPO_URL = 'https://github.com/javierrcastroo/LLM-Juego-de-Tronos.git'

!git clone {REPO_URL}

Cloning into 'LLM-Juego-de-Tronos'...
remote: Enumerating objects: 113, done.
remote: Counting objects: 100% (113/113), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 113 (delta 46), reused 32 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (113/113), 9.23 MiB | 12.81 MiB/s, done.
Resolving deltas: 100% (46/46), done.


In [ ]:
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
drive.mount('/content/drive')


EPUB_PATH = '/content/LLM-Juego-de-Tronos/juego_de_tronos.epub'

SAVE_PATH = Path('/content/drive/MyDrive/NLP/PROYECTO/rag_knowledge_base/bge-m3-finetuned-got')
SAVE_PATH.mkdir(parents=True, exist_ok=True)


def extract_text_from_epub(epub_path):
    print(f"Leyendo el EPUB desde: {epub_path}")
    try:
        book = epub.read_epub(epub_path)
    except Exception as e:
        print(f"Error al leer el EPUB. Detalle: {e}")
        return ""

    text_content = []
    for item in book.get_items_of_type(ebooklib.ITEM_DOCUMENT):
        soup = BeautifulSoup(item.get_body_content(), 'html.parser')
        text_content.append(soup.get_text(separator=' ', strip=True))
    return " ".join(text_content)

full_text = extract_text_from_epub(EPUB_PATH)


print("Tokenizando el texto en oraciones...")
sentences = sent_tokenize(full_text, language='spanish')

# Filtramos: descartamos oraciones muy cortas y excesivamente largas
filtered_sentences = [s for s in sentences if 5 < len(s.split()) < 100]
print(f"Total de oraciones extraídas para entrenar: {len(filtered_sentences)}")

# SimCSE: el input es la misma oración duplicada
train_examples = [InputExample(texts=[s, s]) for s in filtered_sentences]

# Batch size ajustado a 64 para asegurar que no reviente la memoria de la A100
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=64)

print("Descargando y cargando el modelo BAAI/bge-m3...")

# Usamos bfloat16: el formato óptimo para las GPUs A100 (ahorra mucha VRAM)
model = SentenceTransformer('BAAI/bge-m3', model_kwargs={"torch_dtype": torch.bfloat16})

# ¡EL SALVAVIDAS!: Limitamos los tokens a 256.
# Esto evita que el modelo reserve memoria infinita para secuencias anómalas.
model.max_seq_length = 256

train_loss = losses.MultipleNegativesRankingLoss(model)

print("¡Iniciando el Fine-Tuning! (Ahora sí debería ir como la seda)...")
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=1,
    warmup_steps=100,
    show_progress_bar=True,
    optimizer_params={'lr': 2e-5}
)


print(f"Guardando el modelo optimizado en tu Drive: {SAVE_PATH}")
model.save(str(SAVE_PATH))
print("¡Fine-Tuning completado y modelo guardado a salvo!")

Mounted at /content/drive
Leyendo el EPUB desde: /content/LLM-Juego-de-Tronos/juego_de_tronos.epub
Tokenizando el texto en oraciones...
Total de oraciones extraídas para entrenar: 19359
Descargando y cargando el modelo BAAI/bge-m3...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Guardando el modelo optimizado en tu Drive: /content/drive/MyDrive/NLP/PROYECTO/rag_knowledge_base/bge-m3-finetuned-got


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

¡Fine-Tuning completado y modelo guardado a salvo!


### PRUEBA DE EVALUACIÓN

In [ ]:
pruebas = [
    {
        "categoria": "1. Resolución de Alias/Apodos",
        "query": "¿Quién es el Gnomo o Mediohombre?",
        "positivo": "Tyrion Lannister se frotó la nariz. Sabía que a sus espaldas todos en la corte se burlaban de su enanismo.",
        "negativo": "Lord Tywin Lannister observaba desde la mesa del consejo, su mirada fría e imponente no dejaba lugar a dudas sobre quién mandaba."
    },
    {
        "categoria": "2. Recuperación de Eventos Específicos",
        "query": "¿Quién pelea por Tyrion en el Nido de Águilas?",
        "positivo": "El mercenario Bronn dio un paso al frente con una sonrisa torcida, desenvainando su espada para defender al enano en el juicio por combate.",
        "negativo": "Ser Jaime Lannister desenvainó su espada dorada en las calles de Desembarco del Rey, dispuesto a vengar el honor de su familia."
    },
    {
        "categoria": "3. Relaciones y Casas Reales",
        "query": "¿Cuál es el lema de la familia que gobierna Invernalia?",
        "positivo": "Ned miró hacia el norte, sintiendo el frío en el viento. 'Se acerca el invierno', murmuró, recordando las palabras de su Casa.",
        "negativo": "El fuego crepitaba en las antorchas. 'Fuego y Sangre', susurró Viserys, anhelando el momento de recuperar el trono de sus ancestros."
    },
    {
        "categoria": "4. Geografía y Lugares",
        "query": "La gran barrera de hielo del norte que protege el reino.",
        "positivo": "Jon Nieve alzó la vista hacia la inmensa estructura helada que se perdía en el cielo. La Guardia de la Noche lo vigilaría hasta el día de su muerte.",
        "negativo": "La Fortaleza Roja se alzaba imponente sobre la colina de Aegon, un laberinto de piedra y secretos en el corazón de la capital."
    },
    {
        "categoria": "5. Mascotas y Criaturas (Huargos)",
        "query": "El lobo huargo albino de Jon Nieve.",
        "positivo": "Fantasma era el único cachorro blanco de la camada, silencioso y con los ojos rojos como la sangre fresca.",
        "negativo": "Viento Gris gruñó amenazadoramente al Gran Jon Umber, arrancándole dos dedos de un solo mordisco."
    },
    {
        "categoria": "6. Armas Legendarias (Acero Valyrio)",
        "query": "La espada que el Lord Comandante Mormont le regala a Jon.",
        "positivo": "El Viejo Oso le entregó a Garra, cuya empuñadura había sido tallada en piedra con la forma de la cabeza de un lobo.",
        "negativo": "Ned Stark limpió la sangre de Hielo en el bosque de dioses, guardando el imponente espadón en su vaina."
    },
    {
        "categoria": "7. Eventos Trágicos (Caídas)",
        "query": "El accidente que dejó a Bran Stark paralítico.",
        "positivo": "Jaime lo empujó por la ventana de la torre rota murmurando: 'Las cosas que hago por amor'.",
        "negativo": "Lysa Arryn amenazó con lanzar a Tyrion por la Puerta de la Luna para que se estrellara contra las rocas."
    },
    {
        "categoria": "8. Títulos y Facciones (Guardias)",
        "query": "Los caballeros de capa blanca que protegen al Rey.",
        "positivo": "Los siete hermanos juramentados de la Guardia Real destacaban en la sala con sus armaduras inmaculadas.",
        "negativo": "Las capas doradas de la Guardia de la Ciudad marcharon por las calles de Desembarco para mantener el orden."
    },
    {
        "categoria": "9. Relaciones Secretas e Incesto",
        "query": "El verdadero padre del príncipe Joffrey.",
        "positivo": "Ned descubrió por el libro de linajes que el niño de cabello rubio era fruto de la relación secreta de la reina con su hermano Jaime.",
        "negativo": "El rey Robert Baratheon estaba convencido de que su hijo mayor heredaría su fuerza, su martillo y su corona."
    },
    {
        "categoria": "10. Muertes y Ejecuciones (Decapitaciones)",
        "query": "La ejecución del desertor de la Guardia de la Noche.",
        "positivo": "Ned levantó su mandoble y decapitó de un tajo al hombre que aseguraba haber visto a los Caminantes Blancos.",
        "negativo": "Ilyn Payne alzó la gran espada de acero valyrio y cortó la cabeza del traidor frente al Gran Septo de Baelor."
    },
    {
        "categoria": "11. Venenos y Asesinatos",
        "query": "El veneno utilizado para matar a Jon Arryn.",
        "positivo": "Lysa confesó entre lágrimas haber puesto Lágrimas de Lys en el vino de su esposo a petición de Petyr Baelish.",
        "negativo": "El gran maestre Pycelle le dio una dosis de leche de la amapola a Ned para calmar el terrible dolor de su pierna."
    },
    {
        "categoria": "12. Títulos y Liderazgo en el Muro",
        "query": "El líder supremo de la Guardia de la Noche.",
        "positivo": "Jeor Mormont, conocido como el Viejo Oso, observó a los nuevos reclutas desde la galería del Castillo Negro.",
        "negativo": "Ser Barristan Selmy, indignado, arrojó su espada al suelo al ser destituido como líder de la Guardia Real."
    },
    {
        "categoria": "13. Magia y Sobrenatural (Amenazas del Norte)",
        "query": "Los seres de hielo que reviven a los muertos.",
        "positivo": "Los Otros surgieron del bosque nevado en total silencio, sus espadas pálidas brillando bajo la luz de la luna.",
        "negativo": "Los salvajes liderados por Mance Rayder se preparaban para asaltar el Muro y escapar del crudo invierno."
    },
    {
        "categoria": "14. Eventos Políticos (Secuestros)",
        "query": "El arresto de Tyrion Lannister en la posada.",
        "positivo": "Catelyn Tully alzó la voz en la encrucijada y ordenó a los vasallos de su padre que apresaran al enano.",
        "negativo": "Los hombres de Janos Slynt rodearon a Ned Stark en la sala del trono consumando la traición de Meñique."
    },
    {
        "categoria": "15. Geografía y Ciudades",
        "query": "La única ciudad de los señores de los caballos Dothraki.",
        "positivo": "Vaes Dothrak se extendía a los pies de la Madre de las Montañas, enorme y sin murallas que la protegieran.",
        "negativo": "Desembarco del Rey apestaba a humo y suciedad, un laberinto de callejones abarrotados junto a la bahía."
    },
    {
        "categoria": "16. Criaturas Legendarias",
        "query": "El nacimiento de los dragones de Daenerys.",
        "positivo": "Las tres criaturas escamosas se aferraron al cuerpo de la reina cuando el humo de la pira funeraria se disipó.",
        "negativo": "El cráneo gigante de Balerion el Terror Negro acumulaba polvo en las oscuras mazmorras de la Fortaleza Roja."
    },
    {
        "categoria": "17. Entrenamientos y Citas Célebres",
        "query": "La lección del maestro de danza del agua a Arya.",
        "positivo": "Syrio Forel le explicó que solo hay un dios, y a la Muerte solo le decimos una cosa: hoy no.",
        "negativo": "Jon Nieve le regaló a Aguja antes de partir y le aconsejó sonriendo que la clavara por el extremo puntiagudo."
    },
    {
        "categoria": "18. Libros y Documentos Importantes",
        "query": "El libro de las Grandes Casas que lee Ned Stark.",
        "positivo": "Lord Eddard revisó el grueso volumen de linajes de Poniente y notó que la semilla Baratheon siempre es fuerte.",
        "negativo": "Catelyn recibió un mensaje secreto de su hermana Lysa escondido en una pequeña caja de madera tallada."
    },
    {
        "categoria": "19. Batallas y Tácticas Militares",
        "query": "La emboscada donde capturan al Matarreyes.",
        "positivo": "Robb Stark sorprendió al ejército enemigo en el Bosque Susurrante, apresando a Jaime Lannister tras una sangrienta escaramuza.",
        "negativo": "Tywin Lannister colocó a las indisciplinadas tribus de la montaña en la vanguardia durante la Batalla del Forca Verde."
    },
    {
        "categoria": "20. Identidades Ocultas / Mercenarios",
        "query": "Los intentos de asesinato contra Daenerys Targaryen.",
        "positivo": "El mercader de vinos intentó envenenar a la princesa ofreciéndole un barril especial en el mercado dothraki.",
        "negativo": "El asesino del puñal de huesodragón se coló en la habitación del niño comatoso para matarlo en su cama."
    },
    {
        "categoria": "21. Eventos Lúdicos (Torneos)",
        "query": "El gran evento para celebrar el nombramiento de Ned Stark.",
        "positivo": "Las justas y combates del Torneo de la Mano reunieron a los mejores caballeros a las afueras de la capital.",
        "negativo": "El combate singular en el Nido de Águilas decidió si el acusado era inocente o culpable a los ojos de los dioses."
    },
    {
        "categoria": "22. Lore Targaryen e Historia Reciente",
        "query": "El asesinato del Rey Loco.",
        "positivo": "Aerys gritaba '¡Quemadlos a todos!' antes de que el joven león le clavara su espada dorada por la espalda.",
        "negativo": "Robert Baratheon mató al príncipe Rhaegar de un brutal mazazo en el pecho en las aguas del Tridente."
    },
    {
        "categoria": "23. Edificios y Arquitectura",
        "query": "El castillo inexpugnable de la Casa Arryn.",
        "positivo": "El Nido de Águilas se alzaba en lo alto de la montaña Lanza del Gigante, accesible solo a través de mulas y cestos.",
        "negativo": "Invernalia era una antigua fortaleza de granito gris, calentada por aguas termales que corrían por sus muros."
    },
    {
        "categoria": "24. Rituales Dothraki",
        "query": "El banquete nupcial de Daenerys y Drogo.",
        "positivo": "La celebración dothraki duró todo el día entre combates a muerte, bailes salvajes y carne de caballo asada.",
        "negativo": "El banquete de bienvenida en Invernalia estuvo lleno de vino, música y un rey que se reía a carcajadas."
    }
]

In [ ]:
print("Cargando modelo Base (BAAI/bge-m3)...")
base_model = SentenceTransformer('BAAI/bge-m3')

print("\nCargando modelo Fine-Tuned (Juego de Tronos)...")
FINETUNED_PATH = '/content/drive/MyDrive/NLP/PROYECTO/rag_knowledge_base/bge-m3-finetuned-got'
finetuned_model = SentenceTransformer(FINETUNED_PATH)

def evaluar_y_comparar(pruebas, model_base, model_ft):
    print("\n" + "="*60)
    print(" INICIANDO BENCHMARK: BASE vs FINE-TUNED")
    print("="*60)

    margenes_base = []
    margenes_ft = []

    for idx, test in enumerate(pruebas, 1):
        print(f"\n### Prueba {idx}: {test['categoria']}")
        print(f"Pregunta (Query): '{test['query']}'\n")

        # Generar embeddings Base
        emb_query_b = model_base.encode(test['query'], convert_to_tensor=True)
        emb_pos_b = model_base.encode(test['positivo'], convert_to_tensor=True)
        emb_neg_b = model_base.encode(test['negativo'], convert_to_tensor=True)

        # Generar embeddings Fine-Tuned
        emb_query_ft = model_ft.encode(test['query'], convert_to_tensor=True)
        emb_pos_ft = model_ft.encode(test['positivo'], convert_to_tensor=True)
        emb_neg_ft = model_ft.encode(test['negativo'], convert_to_tensor=True)

        # Calcular Similitudes y Márgenes (Positivo - Negativo)
        # Queremos que la similitud con el positivo sea mucho mayor que con el negativo
        sim_pos_b = util.cos_sim(emb_query_b, emb_pos_b).item()
        sim_neg_b = util.cos_sim(emb_query_b, emb_neg_b).item()
        margen_b = sim_pos_b - sim_neg_b
        margenes_base.append(margen_b)

        sim_pos_ft = util.cos_sim(emb_query_ft, emb_pos_ft).item()
        sim_neg_ft = util.cos_sim(emb_query_ft, emb_neg_ft).item()
        margen_ft = sim_pos_ft - sim_neg_ft
        margenes_ft.append(margen_ft)

        # Imprimir resultados de la prueba actual
        print(f"{'Modelo':<15} | {'Sim Positiva':<12} | {'Sim Negativa':<12} | {'Margen (Confianza)':<15}")
        print("-" * 65)
        print(f"{'Base':<15} | {sim_pos_b:<12.4f} | {sim_neg_b:<12.4f} | {margen_b:<15.4f}")
        print(f"{'Fine-Tuned':<15} | {sim_pos_ft:<12.4f} | {sim_neg_ft:<12.4f} | {margen_ft:<15.4f}")

        if margen_ft > margen_b:
            print("🏆 Ganador en esta prueba: FINE-TUNED")
        else:
            print("🏆 Ganador en esta prueba: BASE")

    # Resultados Finales Globales
    print("\n" + "="*60)
    print(" RESULTADOS FINALES GLOBALES (Margen Medio)")
    print("="*60)
    avg_margen_base = np.mean(margenes_base)
    avg_margen_ft = np.mean(margenes_ft)

    print(f"Margen Medio Base:       {avg_margen_base:.4f} (Mayor es mejor)")
    print(f"Margen Medio Fine-Tuned: {avg_margen_ft:.4f} (Mayor es mejor)\n")

    if avg_margen_base > avg_margen_ft:
        print("💡 CONCLUSIÓN: El modelo BASE separa mejor los textos correctos de los incorrectos para RAG.")
    else:
        print("💡 CONCLUSIÓN: El modelo FINE-TUNED ha mejorado la comprensión del contexto específico.")


evaluar_y_comparar(pruebas, base_model, finetuned_model)

Cargando modelo Base (BAAI/bge-m3)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


Cargando modelo Fine-Tuned (Juego de Tronos)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


 INICIANDO BENCHMARK: BASE vs FINE-TUNED

### Prueba 1: 1. Resolución de Alias/Apodos
Pregunta (Query): '¿Quién es el Gnomo o Mediohombre?'

Modelo          | Sim Positiva | Sim Negativa | Margen (Confianza)
-----------------------------------------------------------------
Base            | 0.2228       | 0.2648       | -0.0420        
Fine-Tuned      | 0.0894       | 0.1455       | -0.0562        
🏆 Ganador en esta prueba: BASE

### Prueba 2: 2. Recuperación de Eventos Específicos
Pregunta (Query): '¿Quién pelea por Tyrion en el Nido de Águilas?'

Modelo          | Sim Positiva | Sim Negativa | Margen (Confianza)
-----------------------------------------------------------------
Base            | 0.3424       | 0.2638       | 0.0786         
Fine-Tuned      | 0.1963       | 0.1001       | 0.0962         
🏆 Ganador en esta prueba: FINE-TUNED

### Prueba 3: 3. Relaciones y Casas Reales
Pregunta (Query): '¿Cuál es el lema de la familia que gobierna Invernalia?'

Modelo          | Sim Pos

Hay 10 instancias de 24 que gana el BASE y además, en algunas de ellas el margen del BASE es superior al Fine Tuned. En la mayoría de pruebas que gana el Fine Tuned, el margen de diferencia entre el base y este es casi despreciable. En cambio, hay varios en los que gana el BASE en la que la diferencia entre el margen del Fine Tuned y el Base es altísma.

Por ende, concluimos que es mejor usar el modelo base de embeddings que el Fine Tuned, y podemos concluir que en algunas de las preguntas se lía a la hora de ajustar los pesos.

Por lo tanto, para nuestro RAG usaremos el modelo base de embeddings y no el fine tunneado.